# BASIC IMPORTS

In [1]:
import nltk

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
%matplotlib inline

**Importing WordCloud for text visualisation**

In [3]:
!pip install wordcloud


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from wordcloud import WordCloud

**Importing NLTK for natural language processing**

In [5]:
import nltk
from nltk.corpus import stopwords # for stopwords removal

# downloading NLTK data
nltk.download('stopwords')
nltk.download('punkt') # downloading tokenizer data
nltk.download('punkt_tab') # sentence tokenizer data 

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\anush\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\anush\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\anush\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

**read the csv file**

In [6]:
df = pd.read_csv('spam.csv')
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


**drop the unnecessary columns**

In [7]:
df.drop(columns=['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'],inplace=True)
df.head()

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


**rename the columns**

In [8]:
df.rename(columns={'v1':'target', 'v2':'text'},inplace=True)
df.head()

,target,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


# DATA PREPROCESSING

**label encode the target column**

In [9]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['target'] = le.fit_transform(df['target'])
df.head()


,target,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


**check duplicate values**

In [10]:
df.duplicated().sum()

np.int64(403)

In [11]:
len(df)

5572

**remove duplicate rows**

In [12]:
df = df.drop_duplicates(keep='first')
len(df)

5169

In [13]:
5572-403

5169

# FEATURE ENGINEERING

In [14]:
# import the PorterStemmer for text stemming
from nltk.stem.porter import PorterStemmer
import string
ps = PorterStemmer()


**Text transformation**

In [15]:
def transform_text(text):
    # transform text to lowercase
    text = text.lower()
    # tokenisation using nltk
    text = nltk.word_tokenize(text)
    # removing special characters
    y = []
    for word in text:
        if word.isalnum():
            y.append(word)
    # removing stopwords and punctuation
    text = y[:] # take everything from y
    y.clear()

    # now loop through the tokens and remove stopwords and punctuation
    for word in text:
        if word not in stopwords.words('english') and word not in string.punctuation:
            y.append(word)

    # Stemming using porter stemmer
    text = y[:] # take everythin from y 
    y.clear()
    for word in text:
        y.append(ps.stem(word))

    # Join the list of words with a space in between 
    return " ".join(y)

    
    

**Transform the text**

In [16]:
transform_text('Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...')

'go jurong point crazi avail bugi n great world la e buffet cine got amor wat'

In [17]:
# now do it for the entire text 
df['transformed_text'] = df['text'].apply(transform_text)
df.head()

,target,text,transformed_text
0,0,"Go until jurong point, crazy.. Available only ...",go jurong point crazi avail bugi n great world...
1,0,Ok lar... Joking wif u oni...,ok lar joke wif u oni
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,free entri 2 wkli comp win fa cup final tkt 21...
3,0,U dun say so early hor... U c already then say...,u dun say earli hor u c alreadi say
4,0,"Nah I don't think he goes to usf, he lives aro...",nah think goe usf live around though


**TF-IDF Vectorization**

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features=50)


In [19]:
# input features (convert from sparse matrix to array)
X = tfidf.fit_transform(df['transformed_text']).toarray()
# target column
y = df['target'].values
print(type(X))
print(type(y))

<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


# TRAIN TEST SPLIT

In [20]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=2)

# MODEL TRAINING

In [21]:
!pip install xgboost


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier



**now create objects of all the classes**

In [23]:
svc = SVC(kernel='sigmoid', gamma=1.0)
knc = KNeighborsClassifier()
mnb = MultinomialNB()
dtc = DecisionTreeClassifier(max_depth=5)
lr = LogisticRegression(solver='liblinear',penalty='l1')
rfc = RandomForestClassifier(n_estimators=50, random_state=2)
abc = AdaBoostClassifier(n_estimators=50, random_state=2)
bc = BaggingClassifier(n_estimators=50, random_state=2)
etc = ExtraTreesClassifier(n_estimators=50,random_state=2)
gbdt = GradientBoostingClassifier(n_estimators=50,random_state=2)
xgb = XGBClassifier(n_estimators=50, random_state=2)

In [24]:
clfs = {
    'SVC' : svc,
    'KNN' : knc,
    'NB' : mnb,
    'DT' : dtc,
    'LR' : lr,
    'RF' : rfc,
    'Adaboost' : abc,
    'Bgc' : bc,
    'ETC' : etc,
    'GBDT' : gbdt,
    'xgb' : xgb

}

# MODEL EVALUATION

In [25]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


In [26]:
def train_classifier(clfs,X_train, y_train, X_test,y_test):
    clfs.fit(X_train,y_train)
    y_pred = clfs.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    return accuracy, precision

In [27]:
type(clfs)

dict

In [28]:
accuracy_scores = []
precision_scores = []
for name , clfs in clfs.items():
    current_accuracy, current_precision = train_classifier(clfs, X_train, y_train, X_test, y_test)
    print()
    print("For: ", name)
    print("Accuracy: ", current_accuracy)
    print("Precision: ", current_precision)
    
    accuracy_scores.append(current_accuracy)
    precision_scores.append(current_precision)


For:  SVC
Accuracy:  0.9148936170212766
Precision:  0.711864406779661

For:  KNN
Accuracy:  0.9506769825918762
Precision:  0.8536585365853658

For:  NB
Accuracy:  0.9235976789168279
Precision:  0.9041095890410958

For:  DT
Accuracy:  0.9177949709864603
Precision:  0.7676767676767676

For:  LR
Accuracy:  0.9400386847195358
Precision:  0.8518518518518519


c:\Users\anush\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\anush\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(



For:  RF
Accuracy:  0.9535783365570599
Precision:  0.8688524590163934

For:  Adaboost
Accuracy:  0.9100580270793037
Precision:  0.9411764705882353

For:  Bgc
Accuracy:  0.9506769825918762
Precision:  0.8536585365853658

For:  ETC
Accuracy:  0.9545454545454546
Precision:  0.8699186991869918

For:  GBDT
Accuracy:  0.9342359767891683
Precision:  0.8125

For:  xgb
Accuracy:  0.9526112185686654
Precision:  0.8677685950413223
